# 01 · Clean TUNIZI V2
**Owner:** Aziz · **Input:** `data/raw/tunizi/TUNIZI_V2.txt` · **Output:** `data/interim/tunizi_clean.csv`, `logs/cleaning_log_tunizi.csv`

Steps follow the team spec. Every step is logged (rows before, after, removed) for the cleaning-funnel slide.
Cleaning removes rows and fixes encoding only: no lowercasing, no rewriting words (that is Iskander's conversion step).

In [1]:
# Works both locally (repo root or notebooks/) and in Colab after cloning the repo.
import os, sys
from pathlib import Path
if Path("notebooks").is_dir():
    ROOT = Path(".").resolve()
elif Path("../notebooks").is_dir():
    ROOT = Path("..").resolve()
else:  # Colab: clone first, e.g. !git clone https://github.com/<owner>/tunisian-sentiment.git
    ROOT = Path("/content/tunisian-sentiment")
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Repo root:", ROOT)

Repo root: /home/claude/work/tunisian-sentiment


In [2]:
import pandas as pd
from src.cleaning_utils import fix_text, has_letter, CleaningLog

RAW = ROOT / "data/raw/tunizi/TUNIZI_V2.txt"
OUT = ROOT / "data/interim/tunizi_clean.csv"
LOG = ROOT / "logs/cleaning_log_tunizi.csv"
MAX_WORDS = 200
log = CleaningLog()

## 1. Load and assign ids
Each line is `text;label`. We split on the **last** `;` so a semicolon inside a comment cannot break a row.
Ids are assigned on load, before any row is removed, so every row keeps a stable id through all steps.

In [3]:
from src.load_raw import load_tunizi
df = load_tunizi(ROOT)        # shared loader: same ids in every notebook
log.add("load", len(df), len(df), "TUNIZI_V2.txt, one comment per line")
df.head()

load                                           100000 ->  100000  (removed 0)


,id,text,label_raw
0,tunizi_000001,7yeti ca ya slouma,1
1,tunizi_000002,rabi yehdik aka chenajmou n9oulo,1
2,tunizi_000003,amazing like usual,1
3,tunizi_000004,bonne chance docteur marzouki vive cpr,1
4,tunizi_000005,allah yarhmou we rabi saber ahlo,1


## 2. Labels → `pos` / `neg`, drop neutral
TUNIZI encodes labels as `1` (positive), `-1` (negative), `0` (neutral). Neutral is only ~3.5% and mostly off-topic, so it is dropped (proposal decision).

In [4]:
print(df.label_raw.value_counts())
LABEL_MAP = {"1": "pos", "-1": "neg"}          # "0" = neutral -> dropped
before = len(df)
df = df[df.label_raw.isin(LABEL_MAP)].copy()
df["label"] = df.label_raw.map(LABEL_MAP)
df = df.drop(columns="label_raw")
log.add("drop neutral (label 0)", before, len(df), "label map: 1->pos, -1->neg")

label_raw
1     52711
-1    43767
0      3522
Name: count, dtype: int64
drop neutral (label 0)                         100000 ->   96478  (removed 3522)


## 3. Fix encoding artifacts
`fix_text()` replaces `_x000D_` with a space, strips and collapses repeated spaces (shared with TSAC).

In [5]:
n_x000d = df.text.str.contains("_x000D_", regex=False).sum()
changed = (df.text != df.text.map(fix_text)).sum()
df["text"] = df.text.map(fix_text)
log.add("fix_text (_x000D_, spaces)", len(df), len(df), f"{n_x000d} rows had _x000D_; {changed} rows changed (spaces only)")

fix_text (_x000D_, spaces)                      96478 ->   96478  (removed 0)


## 4. Empty rows and rows without any letter

In [6]:
before = len(df); df = df[df.text != ""]
log.add("remove empty rows", before, len(df))
before = len(df); df = df[df.text.map(has_letter)]
log.add("remove rows without any letter", before, len(df))

remove empty rows                               96478 ->   96478  (removed 0)
remove rows without any letter                  96478 ->   96478  (removed 0)


## 5. Remove comments over 200 words
The proposal found outliers up to 1,721 words, which are not real comments. **Guard:** if this removes more than 0.5% of rows, stop and tell the team.

In [7]:
n_words = df.text.str.split().str.len()
print(n_words.describe(percentiles=[.5, .95, .99, .999]).round(1))
too_long = n_words > MAX_WORDS
share = too_long.mean()
assert share <= 0.005, f"{share:.2%} of rows exceed {MAX_WORDS} words: stop and tell the team"
before = len(df); df = df[~too_long]
log.add(f"remove comments > {MAX_WORDS} words", before, len(df), f"{share:.3%} of rows; longest was {n_words.max()} words")

count    96478.0
mean        10.1
std         15.1
min          2.0
50%          6.0
95%         31.0
99%         62.0
99.9%      141.0
max       1721.0
Name: text, dtype: float64
remove comments > 200 words                     96478 ->   96446  (removed 32)


## 6. Exact duplicates on `text`
Same text + same label → keep the first copy. Same text with opposite labels → drop all copies (we cannot know the true label). The two counts are logged separately.

In [8]:
n_labels = df.groupby("text").label.transform("nunique")
before = len(df); df = df[n_labels == 1]
log.add("remove label conflicts (all copies)", before, len(df))
before = len(df); df = df.drop_duplicates(subset="text", keep="first")
log.add("remove exact duplicates (keep first)", before, len(df))

remove label conflicts (all copies)             96446 ->   96446  (removed 0)
remove exact duplicates (keep first)            96446 ->   96444  (removed 2)


## 7. Add `source` and `split`, check, save

In [9]:
df["source"] = "tunizi"
df["split"] = "train"
df = df[["id", "text", "label", "source", "split"]]

# checks from the spec
assert df.notna().all().all() and (df.text != "").all(), "empty values"
assert set(df.label) <= {"pos", "neg"}, "unexpected label"
assert df.id.is_unique, "duplicate ids"
assert not df.text.duplicated().any(), "duplicate texts left"

OUT.parent.mkdir(parents=True, exist_ok=True); LOG.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False, encoding="utf-8")
log.add("final tunizi_clean.csv", len(df), len(df))
log.save(LOG)
print(df.label.value_counts(normalize=True).round(3))
log.to_frame()

final tunizi_clean.csv                          96444 ->   96444  (removed 0)
label
pos    0.546
neg    0.454
Name: proportion, dtype: float64


,step,rows_before,rows_after,removed,note
0,load,100000,100000,0,"TUNIZI_V2.txt, one comment per line"
1,drop neutral (label 0),100000,96478,3522,"label map: 1->pos, -1->neg"
2,"fix_text (_x000D_, spaces)",96478,96478,0,0 rows had _x000D_; 93 rows changed (spaces only)
3,remove empty rows,96478,96478,0,
4,remove rows without any letter,96478,96478,0,
5,remove comments > 200 words,96478,96446,32,0.033% of rows; longest was 1721 words
6,remove label conflicts (all copies),96446,96446,0,
7,remove exact duplicates (keep first),96446,96444,2,
8,final tunizi_clean.csv,96444,96444,0,


## Notes for the team
- TUNIZI V2 contains **no** `_x000D_` artifacts, empty rows or letter-less rows; those issues (260 `_x000D_`, the empty and letter-less rows) are in **TSAC**.
- Only 2 exact duplicates and no label conflicts in TUNIZI V2.
- A few rows contain Arabic characters; script detection at the merge step decides whether they stay (`arabizi` / `arabic`) or are dropped as `mixed`.